# Perception Test

In [1]:
# ASA Imports
# Notebook Specifics / Temporary Functions
#
import logging

from asa._tools.custom_logging import setup_logging
from asa.affect_model.belief import AffectModel
from asa.core.affect import AffectEvidence, AffectState, Utterance
from asa.core.observers import Event, Observers
from asa.core.representations import EKMAN6
from asa.perception.decode_keyword import EKMAN6_KEYWORDS, KeywordDecoder
from asa.perception.text_console import TextConsole
from asa.runtime import run_agent

setup_logging(level="DEBUG")
log = logging.getLogger("asa.observer.debug")

def read_or_end(prompt: str) -> str:
    """Notebook stand-in for Ctrl-D — no frontend here can send a real EOF."""
    text = input(prompt)
    if text.strip() == ":q":
        raise EOFError
    return text

# def log_event(event: Event) -> None:
#     """Temporary stand-in for the recorder — narrows by type so the fields are checked."""
#     when = event.at.strftime("%H:%M:%S.%f")[:-3]

#     if isinstance(event, Utterance):
#         # log.debug("heard    %s  %r", when, event.text)
#         log.debug("heard    %s  %s  %r", when, event.id, event.text)
#     elif isinstance(event, AffectEvidence):
#         fired = {k: round(v, 2) for k, v in event.affect.values.items() if v}
#         log.debug("evidence %s  %-5s %-16s %s", when, event.target, event.source, fired)
#     elif isinstance(event, AffectState):
#         log.debug("state    %s  other=%s self=%s", when,
#                   dict(event.other.values), dict(event.self_.values))
#     else:
#         log.debug("event    %s  %s  %r", when, event.schema, event)

#     # Full dataclass
#     log.debug("%-12s %s", event.schema, event)

def log_event(event: Event) -> None:
    """Temporary stand-in for the recorder — schema, time, id, then the payload."""
    when = event.at.strftime("%H:%M:%S.%f")[:-3]

    if isinstance(event, Utterance):
        log.debug("%-12s %s  %s  %r", event.schema, when, event.id, event.text)
    elif isinstance(event, AffectEvidence):
        fired = {str(k): round(v, 2) for k, v in event.affect.values.items() if v}
        log.debug("%-12s %s  %s  %-5s %-16s %s",
                  event.schema, when, event.of_input, event.target, event.source, fired)
    elif isinstance(event, AffectState):
        log.debug("%-12s %s  other=%s self=%s",
                  event.schema, when, dict(event.other.values), dict(event.self_.values))
    else:
        log.debug("%-12s %s  %r", event.schema, when, event)

    # Full dataclass
    log.debug("%-12s %s", event.schema, event)
    


# Text Console

In [2]:
# Establish the source, decoder and a simple EKMAN6 keyword decoder
# source = TextConsole()
source = TextConsole(read=read_or_end)
decoder = KeywordDecoder(representation=EKMAN6, table=EKMAN6_KEYWORDS)

# Affectmodel is not implemented and just reports a stub
model = AffectModel()

# Observers is not implemented so just produce a debug log
observers = Observers()
observers.register(log_event)

# Kick-off the agent pipeline
await run_agent(source=source, decoder=decoder, state_writer=model.observe, observers=observers)

DEBUG: asa.observer.debug.log_event.line_49 - utterance/1  16:12:47.192  4f163232abab  'test'
DEBUG: asa.observer.debug.log_event.line_61 - utterance/1  Utterance(text='test', source='input:text_console', intended=None, id='4f163232abab', at=datetime.datetime(2026, 8, 8, 16, 12, 47, 192482, tzinfo=datetime.timezone.utc), schema='utterance/1')
DEBUG: asa.observer.debug.log_event.line_52 - evidence/1   16:12:47.192  4f163232abab  other decoder:rule     {}
DEBUG: asa.observer.debug.log_event.line_61 - evidence/1   AffectEvidence(target=<Target.OTHER: 'other'>, affect=AffectVector(representation='ekman6/1', values={<SixEmotions.ANGER: 'anger'>: 0.0, <SixEmotions.DISGUST: 'disgust'>: 0.0, <SixEmotions.FEAR: 'fear'>: 0.0, <SixEmotions.HAPPINESS: 'happiness'>: 0.0, <SixEmotions.SADNESS: 'sadness'>: 0.0, <SixEmotions.SURPRISE: 'surprise'>: 0.0}), confidence=None, source='decoder:rule', rationale='no keyword matched', computed_from=None, of_input='4f163232abab', at=datetime.datetime(2026, 8, 8,

  + Exception Group Traceback (most recent call last):
  |   File "/Users/stuartgow/PhD Project/Repo asa_research_prototype/.venv/lib/python3.12/site-packages/IPython/core/interactiveshell.py", line 3746, in run_code
  |     await eval(code_obj, self.user_global_ns, self.user_ns)
  |   File "/var/folders/wd/rhzj_w8570g_y66t6j6mh4zw0000gn/T/ipykernel_74219/4239648275.py", line 14, in <module>
  |     await run_agent(source=source, decoder=decoder, state_writer=model.observe, observers=observers)
  |   File "/Users/stuartgow/PhD Project/Repo asa_research_prototype/src/asa/runtime.py", line 60, in run_agent
  |     async with asyncio.TaskGroup() as group:
  |                ^^^^^^^^^^^^^^^^^^^
  |   File "/Users/stuartgow/.local/share/uv/python/cpython-3.12.13-macos-aarch64-none/lib/python3.12/asyncio/taskgroups.py", line 71, in __aexit__
  |     return await self._aexit(et, exc)
  |            ^^^^^^^^^^^^^^^^^^^^^^^^^^
  |   File "/Users/stuartgow/.local/share/uv/python/cpython-3.12.13-

## Test Text

In [3]:
# Get the root path and data paths
#

from pathlib import Path


def repo_root(marker: str = "uv.lock") -> Path:
    """Nearest ancestor of the working directory containing *marker*."""
    start = Path.cwd().resolve()
    for candidate in (start, *start.parents):
        if (candidate / marker).is_file():
            return candidate
    raise FileNotFoundError(f"No {marker} found above {start}")

DATA_IN = repo_root() / "data_in"



In [4]:
# Load the standard benchmark frames
#
# Written by benchmark_datasets.ipynb: source, split, id, text, then one float column per
# axis of the representation. Nothing is reshaped here — that is what the standard shape is
# for. Each file also carries its own provenance in attrs, including any axis its corpus
# never annotated, which the replay source below insists you acknowledge.

import pandas as pd

simple_bench = pd.read_parquet(DATA_IN / "bench_simple_ekman6.parquet")
brighter_bench = pd.read_parquet(DATA_IN / "bench_brighter_eng.parquet")

print(simple_bench.attrs)
print(brighter_bench.attrs)
simple_bench.head()

{'corpus': 'simple-ekman6', 'representation': 'ekman6/1', 'axis_map': {}, 'unannotated_axes': [], 'rows': 20}
{'corpus': 'brighter-eng', 'representation': 'ekman6/1', 'axis_map': {'joy': 'happiness'}, 'unannotated_axes': ['disgust'], 'rows': 8522}


,source,split,id,text,anger,disgust,fear,happiness,sadness,surprise
0,simple-ekman6,all,simple-ekman6_00000,I am so happy,0.0,0.0,0.0,1.0,0.0,0.0
1,simple-ekman6,all,simple-ekman6_00001,I'm absolutely delighted,0.0,0.0,0.0,1.0,0.0,0.0
2,simple-ekman6,all,simple-ekman6_00002,I was gutted,0.0,0.0,0.0,0.0,1.0,0.0
3,simple-ekman6,all,simple-ekman6_00003,I feel miserable today,0.0,0.0,0.0,0.0,1.0,0.0
4,simple-ekman6,all,simple-ekman6_00004,that is absolutely revolting,0.0,1.0,0.0,0.0,0.0,0.0


In [5]:
from asa.core.affect import AffectVector
from asa.core.representations import EKMAN6, AffectRepresentation

# Helper to create an AffectVector
#

def gen_affect_vector(rep: AffectRepresentation, **magnitudes: float) -> AffectVector:
    values = dict.fromkeys(rep.axes, rep.rest)
    for axis, magnitude in magnitudes.items():
        if axis not in values:
            raise ValueError(f"{axis!r} is not an axis of {rep.id}: {rep.axes}")
        values[axis] = magnitude
    return AffectVector(representation=rep.id, values=values)

In [6]:
# An input source that replays a benchmark frame as utterances carrying ground truth
# For evaluating the ASA pipeline

import asyncio
from collections.abc import AsyncIterator, Iterable


class UtterancesFromDF:
    """Replays a benchmark frame as Utterances carrying their intended affect.

    Knows nothing about any particular corpus: it reads ``text`` and the axis columns of the
    representation it is given, which is exactly what the standard benchmark shape
    guarantees are there.

    ``intended`` is the ground truth a decoder is scored against. ``Utterance``'s own
    docstring names a labelled benchmark set as one of only two sources entitled to set it,
    so this is the sanctioned route rather than a test fixture leaking into the runtime.

    ``assume_absent`` names axes whose corpus does not annotate them; any *other* unlabelled
    axis raises. The frames keep an unannotated axis as NaN because ``rest`` is not
    "unknown" — it is the positive claim that the emotion is absent — so somebody has to
    make that claim, and it should be whoever runs the replay, out loud. BRIGHTER-eng needs
    ``assume_absent=("disgust",)``; its ``attrs`` says so.

    ``gap`` is what makes this a *source* rather than a list. A real participant speaks with
    pauses, and an async generator that never awaits would submit the whole frame before the
    consumer ran once — the published order would then be an artefact of the fake rather
    than the pipeline's behaviour.
    """

    def __init__(self,
                 source_df: pd.DataFrame,
                 representation: AffectRepresentation,
                 *,
                 source: str = "input:replay",
                 gap: float = 1.0,
                 assume_absent: Iterable[str] = ()) -> None:
        self._source_df = source_df
        self._rep = representation
        self._source = source
        self._gap = gap
        self._absent = {str(axis) for axis in assume_absent}

    def _intended(self, row: pd.Series) -> AffectVector:
        """The row's labels as a vector; an unmeasured axis is refused unless assumed absent."""
        magnitudes = {}
        for axis in self._rep.axes:
            magnitude = row[axis]
            if pd.isna(magnitude):
                if str(axis) not in self._absent:
                    raise ValueError(f"{axis} is unlabelled in row {row['id']!r} — name it in "
                                     f"assume_absent to assert the corpus means it is absent")
                continue                  # gen_affect_vector's seeded rest supplies it
            magnitudes[str(axis)] = float(magnitude)
        return gen_affect_vector(self._rep, **magnitudes)

    async def events(self) -> AsyncIterator[Utterance]:
        for _, row in self._source_df.iterrows():
            yield Utterance(text=str(row["text"]),
                            source=self._source,
                            intended=self._intended(row))
            await asyncio.sleep(self._gap)

In [7]:
# Establish the replay source, and a simple EKMAN6 keyword decoder
#
# Sliced at the call site rather than with a limit= parameter: the frame is already the
# right object to subset, and 8522 BRIGHTER rows at gap=0.5 would run for 71 minutes. For
# BRIGHTER, the unannotated axis has to be acknowledged:
#   UtterancesFromDF(brighter_bench.head(20), EKMAN6, gap=0.5, assume_absent=("disgust",))

source = UtterancesFromDF(source_df=simple_bench, representation=EKMAN6, gap=0.5)
decoder = KeywordDecoder(representation=EKMAN6, table=EKMAN6_KEYWORDS)

# Affectmodel is not implemented and just reports a stub
model = AffectModel()

# Observers is not implemented so just produce a debug log
observers = Observers()
observers.register(log_event)

# Kick-off the agent pipeline
await run_agent(source=source, decoder=decoder, state_writer=model.observe, observers=observers)

DEBUG: asa.observer.debug.log_event.line_49 - utterance/1  16:27:01.010  0710be551f71  'I am so happy'
DEBUG: asa.observer.debug.log_event.line_61 - utterance/1  Utterance(text='I am so happy', source='input:replay', intended=AffectVector(representation='ekman6/1', values={<SixEmotions.ANGER: 'anger'>: 0.0, <SixEmotions.DISGUST: 'disgust'>: 0.0, <SixEmotions.FEAR: 'fear'>: 0.0, <SixEmotions.HAPPINESS: 'happiness'>: 1.0, <SixEmotions.SADNESS: 'sadness'>: 0.0, <SixEmotions.SURPRISE: 'surprise'>: 0.0}), id='0710be551f71', at=datetime.datetime(2026, 8, 8, 16, 27, 1, 10579, tzinfo=datetime.timezone.utc), schema='utterance/1')
DEBUG: asa.observer.debug.log_event.line_52 - evidence/1   16:27:01.010  0710be551f71  other decoder:rule     {'happiness': 0.7}
DEBUG: asa.observer.debug.log_event.line_61 - evidence/1   AffectEvidence(target=<Target.OTHER: 'other'>, affect=AffectVector(representation='ekman6/1', values={<SixEmotions.ANGER: 'anger'>: 0.0, <SixEmotions.DISGUST: 'disgust'>: 0.0, <SixEmo

  + Exception Group Traceback (most recent call last):
  |   File "/Users/stuartgow/PhD Project/Repo asa_research_prototype/.venv/lib/python3.12/site-packages/IPython/core/interactiveshell.py", line 3746, in run_code
  |     await eval(code_obj, self.user_global_ns, self.user_ns)
  |   File "/var/folders/wd/rhzj_w8570g_y66t6j6mh4zw0000gn/T/ipykernel_74219/1482784971.py", line 19, in <module>
  |     await run_agent(source=source, decoder=decoder, state_writer=model.observe, observers=observers)
  |   File "/Users/stuartgow/PhD Project/Repo asa_research_prototype/src/asa/runtime.py", line 60, in run_agent
  |     async with asyncio.TaskGroup() as group:
  |                ^^^^^^^^^^^^^^^^^^^
  |   File "/Users/stuartgow/.local/share/uv/python/cpython-3.12.13-macos-aarch64-none/lib/python3.12/asyncio/taskgroups.py", line 71, in __aexit__
  |     return await self._aexit(et, exc)
  |            ^^^^^^^^^^^^^^^^^^^^^^^^^^
  |   File "/Users/stuartgow/.local/share/uv/python/cpython-3.12.13-